In [ ]:
from thefuzz import process
import cn2an
import logging
import re
from utils import pre_process, pre_process_without_n

In [ ]:

def find_设置特定目标_rule_base(docs, 一般政策语言):
    一般政策语言 = [re.sub(r'\s+', '', content) for content in 一般政策语言]
    
    logging.getLogger().setLevel(logging.ERROR)
    
    keywords = [
        "(?:^|,)\s*第.*?(章|节|点|条).*?(目标|工作目标|工作重点|发展目标|明确目标|重点目标|重要目标|目标任务|任务|重点任务|主要任务|具体要求|工作要求|主要要求)"
    ]
    rule_1_pattern = re.compile('|'.join(keywords))
    
    # rule_2_keyword_fuzzy = ["总则"]
    
    # rule_3_keyword_fuzzy = ["实现","达到","解决","确保","保证","保障"]
    rule_3_pattern = ["推动.*?目标", "在.*?方面实行", "总则","实现","达到","解决","确保","保证","保障"]
    rule_3_pattern = re.compile('|'.join(rule_3_pattern))
    
    paragraphs = pre_process_without_n(docs)
    matched_paragraphs = []
    for paragraph in paragraphs:
        if any(sentence in paragraph for sentence in 一般政策语言):
            continue
        if rule_1_pattern.search(paragraph):
            matched_paragraphs.append((paragraph.strip(), rule_1_pattern.search(paragraph).group()))
        # elif any(keyword in paragraph for keyword in rule_2_keyword_fuzzy):
        #     matched_paragraphs.add((paragraph.strip(), [keyword for keyword in rule_2_keyword_fuzzy if keyword in paragraph][0]))
        elif rule_3_pattern.search(paragraph):
            matched_paragraphs.append((paragraph.strip(), rule_3_pattern.search(paragraph).group()))
        # else:
        #     best_match = process.extractOne(paragraph, rule_3_keyword_fuzzy)
        #     if best_match[1] >= 60:
        #         matched_paragraphs.add((paragraph.strip(), best_match[0]))
                
    matched_paragraphs_index = []
    for sentence in matched_paragraphs:
        begin_index = docs.find(sentence[0])
        end_index = begin_index + len(sentence[0])
        matched_paragraphs_index.append((begin_index, end_index))
        
    return matched_paragraphs, matched_paragraphs_index
